# Jaxpot Tic-Tac-Toe Colab Quickstart

This notebook is the plug-and-play quickstart for Jaxpot. It clones the repo, installs the project with `uv`, trains a tiny PPO self-play agent on Tic-Tac-Toe, and opens local TensorBoard logs.

**What you should get:** a running Jaxpot training loop, scalar metrics in TensorBoard, checkpoints under `outputs/`, and a compact Hydra config recipe you can adapt to other 1v1 games.

## 0. Choose a runtime

In Colab, open **Runtime -> Change runtime type -> Hardware accelerator -> GPU** before running the notebook. Tic-Tac-Toe is intentionally small enough to run on CPU, so a GPU is helpful but not required.

In [ ]:
# Check whether Colab attached a GPU. CPU-only output is fine for this quickstart.
!nvidia-smi || true

## 1. Clone Jaxpot and install dependencies

Jaxpot is a Python project with dependencies managed by `uv`. After `uv sync`, commands use `uv run ...` so they execute inside the managed project environment instead of whichever Python packages happen to be installed in the notebook runtime.

If you are testing a PR branch, change `BRANCH` before running this cell.

In [ ]:
REPO_URL = "https://github.com/bards-ai/Jaxpot.git"
BRANCH = "main"

!pip -q install uv

import os
from pathlib import Path

repo_dir = Path("/content/Jaxpot")
if not repo_dir.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {repo_dir}
else:
    print(f"Using existing checkout at {repo_dir}")

os.chdir(repo_dir)
print("Working directory:", Path.cwd())

!uv python install 3.12
!uv sync --python 3.12

## 2. Verify JAX sees the runtime

You should see at least one JAX device. On a GPU runtime, one device should mention CUDA. On CPU, the notebook still works; it will just run a little more slowly.

In [ ]:
%%bash
uv run python - <<'PY'
import jax

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
PY

## 3. Prepare the Tic-Tac-Toe experiment

Jaxpot uses Hydra configs under `config/`. To keep this tutorial self-contained, this cell writes the small tutorial-only configs into the Colab checkout at runtime:

- `config/env/tic_tac_toe/default.yaml` selects `pgx.tic_tac_toe.TicTacToe`.
- `config/model/tic_tac_toe_mlp.yaml` selects a small MLP policy/value model.
- `config/logger/tensorboard.yaml` is overwritten in the Colab checkout with a TensorBoard logger wrapped by Jaxpot's `MultiLogger`, which creates a run id for fresh runs across Jaxpot versions.
- `config/experiment/tic_tac_toe/colab.yaml` ties together PPO, the random evaluator, TensorBoard logging, small environment counts, and a short training run.
- `config/logger/wandb_public.yaml` is only for the optional logged-in W&B command later.

The model config does not hard-code observation or action dimensions. `train_selfplay.py` reads those from the environment and injects them when it builds the model.

In [ ]:
from pathlib import Path
from textwrap import dedent

hydra_package = "# " + "@" + "package _global_"

files = {
    "config/env/tic_tac_toe/default.yaml": "_target_: pgx.tic_tac_toe.TicTacToe\n",
    "config/model/tic_tac_toe_mlp.yaml": (
        "_target_: jaxpot.models.architectures.mlp.MLPModel\n"
        "hidden_dims: [64, 64]\n"
    ),
    "config/logger/tensorboard.yaml": dedent("""\
        _target_: jaxpot.loggers.multi.MultiLogger
        _recursive_: false
        logger_configs:
          - _target_: jaxpot.loggers.tensorboard.TensorBoardLogger
            log_dir: ${hydra:runtime.output_dir}/tensorboard
            run_name: ${experiment_name}
        run_id: null
    """),
    "config/logger/wandb_public.yaml": dedent("""\
        _target_: jaxpot.loggers.wandb.WandbLogger
        project_name: "Jaxpot Public"
        run_name: ${experiment_name}
        entity: team-bards-ai
        run_id: null
        tags: ${tags}
    """),
    "config/experiment/tic_tac_toe/colab.yaml": hydra_package + "\n" + dedent("""\
        defaults:
          - override /logger: tensorboard
          - override /model: tic_tac_toe_mlp
          - override /trainer: ppo
          - override /env: tic_tac_toe/default
          - override /eval: random
          - _self_

        tags: ["tic_tac_toe", "colab", "quickstart"]
        experiment_name: "tic_tac_toe_colab_quickstart"

        trainer:
          num_epochs: 2
          batch_size: 512
          auxiliary_losses: []
          clip_eps: 0.2
          entropy_coeff: 0.01
          entropy_coeff_start: 0.03
          entropy_decay_iterations: 100

        seed: 42
        lr: 3e-4
        lr_schedule: "constant"
        multi_gpu: false
        use_target_selfplay: false

        # Keep this small enough for a quick Colab run, including CPU runtimes.
        selfplay_num_envs: 256
        random_num_envs: 128
        league_num_envs: 0
        archive_num_envs: 0
        random_warmup_iters: 0
        league_add_every: 0
        base_unit: 64
        num_steps: 16
        total_iters: 100
        grad_accum_steps: 1
        gamma: 0.99
        gae_lambda: 0.95

        max_grad_norm: 1.0
        save_every: 25
        keep_last_k: 3
        best_checkpoint_top_k: 3
        resume_from: null

        # Faster evaluation for a tutorial notebook.
        eval:
          - _target_: jaxpot.evaluator.random.RandomEvaluator
            eval_every: 20
            num_envs: 256
            num_steps: ${num_steps}
            deterministic: false
            name: eval_vs_random
          - _target_: jaxpot.evaluator.random.RandomEvaluator
            eval_every: 20
            num_envs: 256
            num_steps: ${num_steps}
            deterministic: true
            name: eval_vs_random_deterministic
    """),
}

for file_path, content in files.items():
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content)
    print("ready", file_path)


## 4. Run PPO self-play

This command trains for a short run and logs to TensorBoard by default. The live console output is useful for progress; the durable artifacts are written under `outputs/` in Hydra's run directory.

In [ ]:
!uv run python train_selfplay.py experiment=tic_tac_toe/colab

## 5. Open TensorBoard

TensorBoard reads all runs under `outputs/`, including nested Hydra directories such as `outputs/YYYY-MM-DD/HH-MM-SS/tensorboard`.

If TensorBoard opens but only shows `HPARAMS` and not `Scalars`, stop or rerun the TensorBoard cell after training has written event data. In some notebook/browser sessions TensorBoard needs a fresh start before the Scalars tab appears.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs

## 6. Optional: send a run to public W&B

TensorBoard is the default because it does not require an external account. W&B upload is optional and currently requires that you are logged in with your own W&B account.

If you are logged in, this sends the same experiment to the public Jaxpot project using `config/logger/wandb_public.yaml`: https://wandb.ai/team-bards-ai/Jaxpot%20Public

In [ ]:
# Optional cloud dashboard for logged-in W&B users only.
# !uv run wandb login
# !uv run python train_selfplay.py experiment=tic_tac_toe/colab logger=wandb_public

## 7. Inspect outputs

Each run gets its own Hydra output directory. Look for TensorBoard event files, checkpoints, the resolved config, and the JSON-lines training log.

In [ ]:
!find outputs -maxdepth 5 -type f | sed -n '1,100p'

## Where to go next

Once this works, try changing only the experiment config. For example, this is a tiny Quoridor smoke test, not a serious training run:

```bash
uv run python train_selfplay.py \
  experiment=quoridor/fast \
  logger=tensorboard \
  total_iters=5 \
  selfplay_num_envs=128 \
  random_num_envs=64 \
  league_num_envs=0 \
  archive_num_envs=0 \
  num_steps=32 \
  trainer.batch_size=1024
```

For a real custom game, implement a `pgx.core.Env`, add one env YAML file, choose a model YAML file, and create an experiment YAML like the Tic-Tac-Toe one above.